# DL-03 Residual Connections

- Module: 07 Deep Learning Systems
- Topic: motivating residual blocks through gradient flow and comparing plain versus residual MLPs in PyTorch
- Estimated runtime: 6 to 10 minutes once `torch` and `matplotlib` are installed
- Prerequisites: chain rule through deep compositions, vanishing gradients, and Module 06 backpropagation
- Expected outputs: layerwise gradient-norm plots, loss curves, and a direct illustration of a deep-training pathology and its fix


## Gradient-flow motivation

For a plain block, the Jacobian contribution is just

$$
\frac{\partial h_{\ell + 1}}{\partial h_\ell} = \frac{\partial F_\ell}{\partial h_\ell}.
$$

For a residual block,

$$
h_{\ell + 1} = h_\ell + F_\ell(h_\ell)
\quad \Rightarrow \quad
\frac{\partial h_{\ell + 1}}{\partial h_\ell}
=
I + \frac{\partial F_\ell}{\partial h_\ell}.
$$

The identity path gives gradients a direct route through depth.
This notebook visualizes that difference with a deliberately deep MLP on a synthetic task.


In [ ]:
from __future__ import annotations

import random

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 23
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")


In [ ]:
def make_dataset(n_samples: int = 2200):
    x = torch.empty(n_samples, 2).uniform_(-2.5, 2.5)
    radial = x[:, 0] ** 2 + 0.8 * x[:, 1] ** 2
    stripe = torch.sin(2.8 * x[:, 0]) + 0.5 * torch.cos(3.5 * x[:, 1])
    y = ((radial + stripe) > 2.2).long()
    x = x + 0.1 * torch.randn_like(x)
    return x, y


x, y = make_dataset()
permutation = torch.randperm(len(x))
train_idx = permutation[:1700]
test_idx = permutation[1700:]

train_dataset = TensorDataset(x[train_idx], y[train_idx])
test_dataset = TensorDataset(x[test_idx], y[test_idx])


class PlainDeepMLP(nn.Module):
    def __init__(self, width: int = 64, depth: int = 18):
        super().__init__()
        blocks = []
        blocks.append(nn.Linear(2, width))
        blocks.append(nn.ReLU())
        for _ in range(depth - 1):
            blocks.append(nn.Linear(width, width))
            blocks.append(nn.ReLU())
        self.features = nn.Sequential(*blocks)
        self.head = nn.Linear(width, 2)

    def forward(self, x):
        return self.head(self.features(x))


class ResidualBlock(nn.Module):
    def __init__(self, width: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, width),
        )
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.activation(x + self.block(x))


class ResidualMLP(nn.Module):
    def __init__(self, width: int = 64, depth: int = 9):
        super().__init__()
        self.stem = nn.Sequential(nn.Linear(2, width), nn.ReLU())
        self.blocks = nn.Sequential(*[ResidualBlock(width) for _ in range(depth)])
        self.head = nn.Linear(width, 2)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        return self.head(x)


def collect_linear_grad_norms(model: nn.Module):
    norms = []
    for module in model.modules():
        if isinstance(module, nn.Linear) and module.weight.grad is not None:
            norms.append(module.weight.grad.detach().norm().item())
    return norms


def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0
    loss_fn = nn.CrossEntropyLoss()
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device))
            loss = loss_fn(logits, yb.to(device))
            correct += (logits.argmax(dim=1) == yb.to(device)).sum().item()
            total += len(xb)
            total_loss += loss.item() * len(xb)
    return total_loss / total, correct / total


def train(model: nn.Module, epochs: int = 30, lr: float = 3e-3):
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=256)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    accuracies = []
    gradient_snapshots = []

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            if epoch in {0, 4, epochs - 1}:
                gradient_snapshots.append((epoch, collect_linear_grad_norms(model)))
            optimizer.step()
        test_loss, test_acc = evaluate(model, test_loader)
        losses.append(test_loss)
        accuracies.append(test_acc)
    return losses, accuracies, gradient_snapshots


In [ ]:
plain_model = PlainDeepMLP().to(device)
residual_model = ResidualMLP().to(device)

plain_losses, plain_acc, plain_grads = train(plain_model)
residual_losses, residual_acc, residual_grads = train(residual_model)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(plain_losses, label="plain")
axes[0].plot(residual_losses, label="residual")
axes[0].set_title("Test loss")
axes[1].plot(plain_acc, label="plain")
axes[1].plot(residual_acc, label="residual")
axes[1].set_title("Test accuracy")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for epoch, grads in plain_grads[:3]:
    axes[0].plot(grads, marker="o", label=f"epoch {epoch}")
for epoch, grads in residual_grads[:3]:
    axes[1].plot(grads, marker="o", label=f"epoch {epoch}")

axes[0].set_title("Plain network gradient norms by linear layer")
axes[1].set_title("Residual network gradient norms by linear layer")
for ax in axes:
    ax.set_xlabel("Linear-layer index")
    ax.set_ylabel("Gradient norm")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()


## Interpretation

The plain network in this notebook is intentionally deep enough to make optimization awkward.
Typical outcomes are:

- slower loss reduction for the plain network;
- weaker or less uniform gradient norms in early layers of the plain network;
- faster convergence and more even gradient transport in the residual model.

This is the key conceptual point of residual design.
Residual blocks do not just add parameters.
They alter the optimization landscape by preserving a direct signal path.


## Exercises

1. Increase the plain-network depth from `18` to `30`. How quickly does training degrade?
2. Add `LayerNorm` inside the residual block and compare convergence.
3. Clip the gradients at norm `1.0` and check whether the plain network becomes less erratic.
4. Replace `ReLU` with `Tanh`. Does the residual advantage become even clearer?
